### 2-3. Train on COCO dataset **(Problem h, 20 pts)**

In [ ]:
Data preparation

In [ ]:
# local machine
# !unzip -n data.zip
# root_path = "."

# colab
from google.colab import drive
drive.mount('/content/drive')

# modify path
root_path = "/content/drive/MyDrive/2026_sp_dl"
data_path = f"{root_path}/data.zip"

!unzip -n {data_path} -d {root_path}

In [ ]:
import os
import json
import math
import random
import torchvision.transforms as T

from PIL import Image
from tqdm import tqdm
from datetime import datetime
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

#### Dataset class

In [ ]:
class CocoDataset(Dataset):
    def __init__(self, image_root, annotation_file, is_train=True):
        with open(annotation_file, "r") as f:
            data = json.load(f)

        self.images = data["images"]
        self.annotations = data["annotations"]
        self.image_root = image_root

        self.id_to_caps = {}
        for ann in self.annotations:
            self.id_to_caps.setdefault(ann["image_id"], []).append(ann["caption"])

        self.transform = T.Compose([
            T.Resize((224, 224)),
            T.ToTensor(),
            T.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

        self.is_train = is_train


    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_id = img_info["id"]

        img_path = os.path.join(self.image_root, img_info["file_name"])
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)

        if self.is_train:
            caption = random.choice(self.id_to_caps[img_id])
        else:
            caption = self.id_to_caps[img_id][0]
        return image, caption

You may add image augmentations.
Note that validation/test procedures must use the original transform function.

In [ ]:
train_dataset = CocoDataset(
    f"{root_path}/data/train/images",
    f"{root_path}/data/train/annotations/captions_train.json",
    is_train=True
)

valid_dataset = CocoDataset(
    f"{root_path}/data/valid/images",
    f"{root_path}/data/valid/annotations/captions_valid.json",
    is_train=False
)

# TODO: You may apply image augmentations during training
train_dataset.transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.5, 1.0), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

#### Text encoder
We use pretrained DistilBERT text encoder for this assignment.

SANH, et al. DistilBERT, a distilled version of BERT: smaller,
faster, cheaper and lighter, EMC^2: 5th Edition Co-located with NeurIPS’19.

(https://arxiv.org/pdf/1910.01108)

In [ ]:
# Do not modify this block

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
text_model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)

for p in text_model.parameters():
    p.requires_grad = False

text_dim = text_model.config.hidden_size  # 768

#### Training

Using the ViT implemented above, perform contrastive training.

The following constraints must be strictly followed:
- Do not use any data other than the provided training dataset (e.g., validation set or external data).
- Do not use pretrained weights for the vision encoder.
- Do not train the provided text encoder.

Except for the above restrictions, all other design choices are free (e.g., model architecture (ViT design), number of training epochs, optimizer, learning rate scheduler, and image augmentations).

In [ ]:
vision_encoder = ViT(
    img_size=224,
    patch_size=16,
    embed_dim=256,      # 16 → 256
    depth=6,            # 1 → 6
    num_heads=8,        # 1 → 8
    mlp_ratio=4.0,      # 1.0 → 4.0
).to(device)
vision_proj = nn.Linear(vision_encoder.embed_dim, text_dim).to(device)

In [ ]:
TRAIN_BATCH_SIZE = 128
TRAIN_EPOCH = 20
SAVE_FREQ = 2
SERIAL = datetime.now().strftime("%y%m%d-%H%M%S")

In [ ]:
logit_scale = nn.Parameter(torch.ones([], device=device) * math.log(1 / 0.07))

optimizer = torch.optim.AdamW(
    list(vision_encoder.parameters()) +
    list(vision_proj.parameters()) +
    [logit_scale],
    lr=5e-4,
    weight_decay=0.05,
    betas=(0.9, 0.98),
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=TRAIN_EPOCH, eta_min=1e-6
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=4
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=64,
    shuffle=False
)

#### Evaluation

The following metrics are used to evaluate image-text retrieval performance:

- R@1/R@10 (Recall@1/Recall@10): The proportion of samples where the correct text is within the top 1/10 retrieved results. Higher is better.

- Mean/Median Rank: The average/median rank of the correct text across all samples. Lower is better.

- **Mean Top-% Position**: The average percentile position of the correct text among all candidates. Lower is better.

Grading will be based on the validation set **Mean Top-% Position** performance as follows:
- ≤ 20% → 5 pts  
- ≤ 15% → 10 pts  
- ≤ 12% → 15 pts  
- ≤ 10% → 20 pts  

However, if the evaluation on a hidden test set differs from the validation score by more than 50%, the test score will be used for grading.

In [ ]:
# Do not modify this block

@torch.no_grad()
def evaluate(loader, vision_encoder, vision_proj,
             text_model, tokenizer, device):

    vision_encoder.eval()
    vision_proj.eval()

    all_img = []
    all_txt = []

    for images, captions in loader:
        images = images.to(device)

        inputs = tokenizer(
            list(captions),
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        txt_out = text_model(**inputs)
        txt_feat = txt_out.last_hidden_state[:, 0]

        img_feat = vision_proj(vision_encoder(images))

        img_feat = F.normalize(img_feat, dim=-1)
        txt_feat = F.normalize(txt_feat, dim=-1)

        all_img.append(img_feat)
        all_txt.append(txt_feat)

    all_img = torch.cat(all_img)
    all_txt = torch.cat(all_txt)

    sim = all_img @ all_txt.T
    ranks = sim.argsort(dim=-1, descending=True)

    correct = torch.arange(len(sim), device=sim.device)

    r1 = (ranks[:, :1] == correct.unsqueeze(1)).any(dim=1).float().mean()
    r10 = (ranks[:, :10] == correct.unsqueeze(1)).any(dim=1).float().mean()

    correct_positions = (ranks == correct.unsqueeze(1)).nonzero(as_tuple=False)[:, 1]

    mean_rank = correct_positions.float().mean() + 1
    median_rank = correct_positions.float().median() + 1

    N = len(sim)
    percentiles = (correct_positions.float() + 1) / N * 100
    mean_percentile = percentiles.mean()

    print(f"Validation R@1:  {r1.item():.4f}")
    print(f"Validation R@10: {r10.item():.4f}")
    print(f"Mean Rank: {mean_rank.item():.2f}")
    print(f"Median Rank: {median_rank.item():.2f}")
    print(f"Mean Top-% Position: {mean_percentile.item():.2f}%")

    return mean_percentile.item()

#### Save/Load checkpoint

In [ ]:
def save_checkpoint(save_dir, vision_encoder, vision_proj):
    os.makedirs(save_dir, exist_ok=True)

    torch.save(vision_encoder.state_dict(), os.path.join(save_dir, "vision_encoder.pt"))
    torch.save(vision_proj.state_dict(), os.path.join(save_dir, "vision_proj.pt"))

    print(f"Saved checkpoint to {save_dir}")

def load_checkpoint(load_dir, vision_encoder, vision_proj, device="cpu"):
    ve_path = os.path.join(load_dir, "vision_encoder.pt")
    vp_path = os.path.join(load_dir, "vision_proj.pt")

    vision_encoder.load_state_dict(torch.load(ve_path, map_location=device))
    vision_proj.load_state_dict(torch.load(vp_path, map_location=device))

    vision_encoder.to(device)
    vision_proj.to(device)

    vision_encoder.eval()
    vision_proj.eval()

    print(f"Loaded checkpoint from {load_dir}")

In [ ]:
total_steps = len(train_loader) * TRAIN_EPOCH

best_score = evaluate(valid_loader, vision_encoder, vision_proj, text_model, tokenizer, device)
best_epoch = 0
val_history = []

for epoch in range(TRAIN_EPOCH):

        vision_encoder.train()
        vision_proj.train()

        total_loss = 0

        for images, captions in tqdm(train_loader):

            images = images.to(device)

            inputs = tokenizer(
                list(captions),
                padding=True,
                truncation=True,
                return_tensors="pt"
            ).to(device)

            with torch.no_grad():
                txt_out = text_model(**inputs)
                txt_feat = txt_out.last_hidden_state[:, 0]

            img_feat = vision_proj(vision_encoder(images))

            # TODO: You can choose your loss function between CLIP & SigLIP
            loss = clip_loss(img_feat, txt_feat, logit_scale)
            # loss = siglip_loss(img_feat, txt_feat, logit_scale)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(vision_encoder.parameters()) +
                list(vision_proj.parameters()),
                max_norm=1.0
            )
            optimizer.step()

            with torch.no_grad():
                logit_scale.clamp_(max=math.log(100))

            total_loss += loss.item()

        scheduler.step()  # epoch 끝에 한 번 호출

        print(
            f"\nEpoch {epoch} | "
            f"Loss: {total_loss/len(train_loader):.4f} | "
        )

        valid_score = evaluate(valid_loader, vision_encoder, vision_proj, text_model, tokenizer, device)
        val_history.append(valid_score)

        if (epoch + 1) % SAVE_FREQ == 0:
            save_dir = f"{root_path}/checkpoints/exp-{SERIAL}/epoch-{epoch + 1}_score-{valid_score:.3f}"
            save_checkpoint(save_dir, vision_encoder, vision_proj)
        if valid_score < best_score:
            save_dir = f"{root_path}/checkpoints/exp-{SERIAL}/best"
            save_checkpoint(save_dir, vision_encoder, vision_proj)

            best_epoch = epoch
            best_score = valid_score